In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [2]:
duck.sql(
    """
    create or replace table wochenliste as 
    select 
        Firmenname,
        "ID Nummer" as easybill_id,
        Standort 
    from read_csv('data/2026_KW14_KW15_Wochenliste.xlsm - Kunden.csv', header=true)
    """
)

In [10]:
duck.sql(
    """
    create or replace table uber_details as
    select 
        * 
    from read_csv('data/Überregionale Kunden_sales_zu ergänzen-2.xlsx - ÜR Kunden (3).csv')
    """
)

In [11]:
import csv
import glob
import os
import re
import unicodedata

col_names = [
    'col0','col1','col2','row_num','unternehmen','kd_nr','typ_amed','typ_asi',
    'arzt','fasi','betreuungsart','direkte_leistung','mitarbeiter','bg',
    'vertragsbeginn','start_aktuelle_periode','vertragsstunden','davon_amed',
    'davon_asi','davon_apsy','zusaetzl_amed','zusaetzl_asi','zusaetzl_apsy',
    'vertragsstunden_direkt','amed_direkt','asi_direkt','apsy_direkt',
    'stunden_erbracht','amed_erbracht','asi_erbracht','apsy_erbracht',
    'fahrtzeit_erbracht','stunden_offen','amed_offen','asi_offen','apsy_offen',
    'beleistung_gesamt','beleistung_amed','beleistung_asi',
    'gesamt','spezifisch_amed','spezifisch_asi','col42'
]

# Priority when the same (city, kd_nr) shows up in several extracts — highest priority wins:
#   1. new "PADOA EXPORT" files (only Rostock has been promoted to that format so far)
#   2. "Stundenerfassung Dashboard_<date>(city)" extracts, most recent date first
#      (e.g. 20260702 beats the older 20260409). Column positions are identical across all
#      formats; the newer dashboard exports just ship ~39 cols (padded to 43 below).
# Clients present only in an older / lower-priority file are still kept (update + insert semantics).
padoa_files = sorted(f for f in glob.glob('basic_care_details_data/PADOA EXPORT_*.csv') if 'Rostock' in f)
dashboard_files = sorted(
    glob.glob('basic_care_details_data/Stundenerfassung Dashboard_*.csv'),
    key=lambda f: re.search(r'(\d{8})', os.path.basename(f)).group(1),
    reverse=True,  # newest extract first so it wins the dedup
)
csv_files = padoa_files + dashboard_files

seen = set()  # (city, kd_nr) — earlier (higher-priority) files win
all_rows = []
for f in csv_files:
    # macOS filenames are NFD; normalize to NFC so umlauts match data from other sources
    city = unicodedata.normalize('NFC', os.path.basename(f).split('(')[1].split(')')[0])
    # PADOA + recent dashboard exports ship as UTF-8 with a BOM (umlauts preserved);
    # older dashboard exports are latin-1 — detect by the BOM rather than by filename
    enc = 'utf-8-sig' if open(f, 'rb').read(3) == b'\xef\xbb\xbf' else 'latin-1'
    with open(f, encoding=enc) as fh:
        reader = list(csv.reader(fh, delimiter=';'))
    for r in reader[5:]:
        # some exports ship < 43 cols; pad so col positions stay consistent
        if len(r) < 43:
            r = r + [''] * (43 - len(r))
        # require row_num (col 3) numeric AND kd_nr (col 5) non-empty —
        # excludes summary/total rows in the spreadsheet
        if r[3].strip().isdigit() and r[5].strip():
            key = (city, r[5].strip())
            if key in seen:
                continue
            seen.add(key)
            all_rows.append([city] + r[:43])

duck.sql('CREATE OR REPLACE TABLE raw_details (standort VARCHAR, ' + ', '.join([f'{c} VARCHAR' for c in col_names]) + ')')
duck.executemany('INSERT INTO raw_details VALUES (' + ','.join(['?'] * 44) + ')', all_rows)

print(f"Loaded {len(all_rows)} rows from {len(csv_files)} files")

Loaded 1063 rows from 18 files


In [12]:
# normalize German short-date strings ("1/ Jan 25", "15/ Jul 25", "1. Feb 26") to ISO "YYYY-MM-DD";
# newer dashboard exports use a dot ("1. Feb 26"), older ones a slash ("1/ Jan 25") — accept both.
# leave NULL, "#VALEUR!" and anything that doesn't match untouched
duck.sql(r"""
create or replace macro norm_date(x) as
case
  when regexp_full_match(trim(x), '\d{1,2}[/.]\s*(Jan|Feb|Mrz|Apr|Mai|Jun|Jul|Aug|Sep|Okt|Nov|Dez)\s+\d{2,4}') then
      (case when length(regexp_extract(trim(x),'(\d{2,4})$',1))=2 then '20' else '' end)
      || regexp_extract(trim(x),'(\d{2,4})$',1) || '-' ||
      (case regexp_extract(trim(x),'[/.]\s*(\w+)\s',1)
        when 'Jan' then '01' when 'Feb' then '02' when 'Mrz' then '03' when 'Apr' then '04'
        when 'Mai' then '05' when 'Jun' then '06' when 'Jul' then '07' when 'Aug' then '08'
        when 'Sep' then '09' when 'Okt' then '10' when 'Nov' then '11' when 'Dez' then '12' end)
      || '-' || lpad(regexp_extract(trim(x),'(\d{1,2})[/.]',1),2,'0')
  else x
end
""")

duck.sql(
    """
    create or replace table full_basic_care as
    with uber_mothers as (
        select distinct left("Wochenliste ID"::varchar, 9) as mother_id
        from uber_details
    ),
    merged as (
        select
            coalesce(left("Wochenliste ID"::varchar, 9), left(trim(rd.kd_nr), 9)) as mother_client_id,
            coalesce("Wochenliste ID"::varchar, trim(rd.kd_nr)) as child_client_id,
            ud."Wochenliste ID" as ud_id,
            trim(rd.unternehmen) as unternehmen,
            ud.Kunde as kunde,
            coalesce(ud."BAS Standort", rd.standort) as Standort,
            ud.Anschrift,
            trim(rd.arzt) as arzt,
            trim(rd.fasi) as fasi,
            trim(rd.typ_amed) as typ_amed,
            trim(rd.typ_asi) as typ_asi,
            trim(rd.betreuungsart) as betreuungsart,
            trim(rd.direkte_leistung) as direkte_leistung,
            -- employee count for überregional clients comes from the ÜR Kunden list (per Anschrift);
            -- fall back to the Stundenerfassung total only for non-über clients
            coalesce(ud.Mitarbeitende::varchar, trim(rd.mitarbeiter)) as mitarbeiter,
            trim(rd.bg) as bg,
            trim(rd.vertragsbeginn) as vertragsbeginn,
            trim(rd.start_aktuelle_periode) as start_aktuelle_periode,
            -- sold/contract hours for überregional clients also come from the ÜR Kunden list (per Anschrift);
            -- fall back to the Stundenerfassung total only for non-über clients (ÜR has no APSY split)
            coalesce(ud."Stunden gesamt", trim(rd.vertragsstunden)) as vertragsstunden,
            coalesce(ud."Stunden AM", trim(rd.davon_amed)) as davon_amed,
            coalesce(ud."Stunden AS", trim(rd.davon_asi)) as davon_asi,
            trim(rd.davon_apsy) as davon_apsy
        from uber_details ud
        full outer join raw_details rd
            on "Wochenliste ID"::varchar = trim(rd.kd_nr)
    )
    select
        m.mother_client_id,
        m.child_client_id,
        -- prefer the per-Anschrift ÜR name, then the clean easybill contact name;
        -- raw_details.unternehmen is last because some Stundenerfassung CSVs ship with U+FFFD
        -- replacement chars where umlauts should be (umlauts lost during the xlsm→csv export)
        coalesce(m.kunde, c."Kontakt: Firma", m.unternehmen) as firm_name,
        m.Standort,
        m.Anschrift,
        m.arzt, m.fasi, m.typ_amed, m.typ_asi, m.betreuungsart, m.direkte_leistung,
        m.mitarbeiter, m.bg,
        norm_date(m.vertragsbeginn) as vertragsbeginn,
        norm_date(m.start_aktuelle_periode) as start_aktuelle_periode,
        m.vertragsstunden, m.davon_amed, m.davon_asi, m.davon_apsy
    from merged m
    left join pg.easybill.contacts c
        on m.mother_client_id = c."Kontakt: Kundennummer"
    where m.child_client_id is not null
      and (
          m.ud_id is not null  -- always keep uber_details rows
          or m.mother_client_id not in (select mother_id from uber_mothers)  -- only keep raw rows when mother absent from uber
      )
    order by m.mother_client_id, m.child_client_id
    """
)

In [13]:
duck.sql(
    """
    select sum(mitarbeiter::numeric) from full_basic_care where mother_client_id = '130001323' 
    """
)

┌─────────────────────────────────────────┐
│ sum(CAST(mitarbeiter AS DECIMAL(18,3))) │
│              decimal(38,3)              │
├─────────────────────────────────────────┤
│                                 355.000 │
└─────────────────────────────────────────┘

In [14]:
duck.sql(
    """
    select * from full_basic_care
    """
)
#.to_csv('full_basic_care.csv')

┌──────────────────┬─────────────────┬─────────────────────────────────────────────────────────────┬────────────┬────────────────────────────────────────┬───────────┬────────────────┬──────────┬─────────┬───────────────┬──────────────────┬─────────────┬─────────┬────────────────┬────────────────────────┬─────────────────┬────────────┬───────────┬────────────┐
│ mother_client_id │ child_client_id │                          firm_name                          │  Standort  │               Anschrift                │   arzt    │      fasi      │ typ_amed │ typ_asi │ betreuungsart │ direkte_leistung │ mitarbeiter │   bg    │ vertragsbeginn │ start_aktuelle_periode │ vertragsstunden │ davon_amed │ davon_asi │ davon_apsy │
│     varchar      │     varchar     │                           varchar                           │  varchar   │                varchar                 │  varchar  │    varchar     │ varchar  │ varchar │    varchar    │     varchar      │   varchar   │ varchar │    varchar  

In [15]:
from datetime import datetime

# back up the current table before overwriting it — timestamped so each run keeps its own copy.
# skipped on the very first run, when the target table doesn't exist yet.
backup_name = f"full_basic_care_backup_{datetime.now():%Y%m%d_%H%M%S}"
try:
    duck.sql(f"""
        create table pg.bas_firms.{backup_name} as
        select * from pg.bas_firms.full_basic_care
    """)
    print(f"Backed up existing table to pg.bas_firms.{backup_name}")
except Exception as e:
    print(f"No backup made (table may not exist yet): {e}")

duck.sql(
    """
    create or replace table pg.bas_firms.full_basic_care as
    select * from full_basic_care
    """
)

Backed up existing table to pg.bas_firms.full_basic_care_backup_20260703_143539
